# 05 — Statistics & Hypotheses

Implements Stage 5 (`09_ETL_AND_ANALYSIS_ENGINE.md`) exactly as `04_STATISTICAL_ANALYSIS_PLAN.md` specifies: simple linear trend (year -> metric) and Pearson correlation (Spearman as a secondary check), significance at **alpha = 0.05**. No multiple regression, no multiple-comparison correction, no residual diagnostics.

> All calculations are imported from `src/`; this notebook only orchestrates and reports (`11_CODE_STRUCTURE.md`).

**Bird data is the REAL eBird extract (EBD v1.16, IN-GJ, May 2026). Environmental values are still MOCK placeholders (GEE not authenticated), so H1 / temperature results are NOT real yet.**

In [1]:
# --- Setup: make src/ importable (works whether cwd is repo root or notebooks/) ---
import os, sys, datetime, platform
_root = os.getcwd()
while not os.path.exists(os.path.join(_root, "requirements.txt")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
REPO_ROOT = _root
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np, pandas as pd, scipy
import matplotlib
matplotlib.use("Agg")            # headless: write figure files, no GUI needed
import matplotlib.pyplot as plt

from src import load_and_clean, observer_effort, migration_metrics
from src import environmental_data as envmod
from src import statistics as stats_
from src import validation as V

FIG_DIR = os.path.join(REPO_ROOT, "outputs", "figures")
TAB_DIR = os.path.join(REPO_ROOT, "outputs", "tables")
os.makedirs(FIG_DIR, exist_ok=True); os.makedirs(TAB_DIR, exist_ok=True)

# Bird data is the REAL eBird extract; environmental values are still MOCK
# placeholders (GEE not authenticated), so H1 / temperature results are not real.
DATA_NOTE = "Bird data: eBird EBD v1.16 (IN-GJ, May 2026). ENV = MOCK (GEE not authenticated)"
STUDY_PERIOD = "2010-2025"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
print("setup complete; REPO_ROOT =", REPO_ROOT)

setup complete; REPO_ROOT = /home/tops/Documents/BirdSense


In [2]:
# --- Reproducibility header (04_STATISTICAL_ANALYSIS_PLAN.md) ---
print("Dataset version :", "eBird EBD v1.16 (IN-GJ, relMay-2026); ENV=MOCK placeholder")
print("Analysis date   :", datetime.date.today().isoformat())
print("Python          :", platform.python_version())
print("pandas", pd.__version__, "| numpy", np.__version__,
      "| scipy", scipy.__version__, "| matplotlib", matplotlib.__version__)

Dataset version : eBird EBD v1.16 (IN-GJ, relMay-2026); ENV=MOCK placeholder
Analysis date   : 2026-07-14
Python          : 3.10.12
pandas 2.3.3 | numpy 2.2.6 | scipy 1.15.3 | matplotlib 3.10.9


In [3]:
# --- Run Stages 1-4 from src (no metric logic here; all imported) ---
stage1     = load_and_clean.run_stage1()
effort     = observer_effort.compute_observer_effort(stage1.clean_observations, stage1.clean_checklists)
metrics    = migration_metrics.compute_migration_metrics(stage1.clean_observations, stage1.clean_checklists)
annual_env = envmod.build_annual_environmental(mock=True)   # FAKE env values (pre-GEE)
print("metrics", metrics.shape, "| effort", effort.shape, "| annual_env", annual_env.shape)

!! MOCK ENVIRONMENTAL DATA: values are FAKE (deterministic), NOT real GEE output. Never use in the paper. Set mock=False (with authenticated GEE) for real values.
metrics (192, 14) | effort (192, 6) | annual_env (16, 5)


## Trend analysis (H2): confirmed arrival & departure vs year
Per-species linear regression; low-confidence species-years excluded (`02_METRICS_METHODOLOGY.md` sec 1).

In [4]:
arrival_trend = stats_.species_metric_trend(metrics, "first_arrival")
arrival_trend

,species,common,habitat,slope,intercept,r_squared,p_value,n
0,Anas acuta,Northern Pintail,wetland,-0.523529,1059.345588,0.424065,0.006286,16
1,Spatula clypeata,Northern Shoveler,wetland,-0.366176,741.073529,0.416575,0.006929,16
2,Spatula querquedula,Garganey,wetland,-0.973529,1969.345588,0.333926,0.019056,16
3,Mareca penelope,Eurasian Wigeon,wetland,-0.714706,1445.669118,0.600946,0.000419,16
4,Aythya ferina,Common Pochard,wetland,-1.194118,2415.257353,0.430657,0.005764,16
5,Anser indicus,Bar-headed Goose,grassland_dryland,-1.644118,3325.132353,0.570226,0.000720,16
6,Anser anser,Greylag Goose,wetland,-0.886765,1793.360294,0.464620,0.003638,16
7,Grus grus,Common Crane,grassland_dryland,-0.411765,833.360294,0.390166,0.009688,16
8,Grus virgo,Demoiselle Crane,grassland_dryland,-0.867647,1755.227941,0.575182,0.000661,16
9,Phoenicopterus roseus,Greater Flamingo,wetland,-0.544118,1101.007353,0.371446,0.012199,16


In [5]:
departure_trend = stats_.species_metric_trend(metrics, "last_departure")
departure_trend

,species,common,habitat,slope,intercept,r_squared,p_value,n
0,Anas acuta,Northern Pintail,wetland,0.927941,-1509.808824,0.322629,0.021713,16
1,Spatula clypeata,Northern Shoveler,wetland,0.919118,-1491.882353,0.295517,0.029516,16
2,Spatula querquedula,Garganey,wetland,1.041176,-1739.198529,0.357928,0.014359,16
3,Mareca penelope,Eurasian Wigeon,wetland,1.039706,-1735.669118,0.385687,0.010243,16
4,Aythya ferina,Common Pochard,wetland,0.979412,-1613.963235,0.355277,0.014821,16
5,Anser indicus,Bar-headed Goose,grassland_dryland,14.416176,-28763.698529,0.341914,0.017358,16
6,Anser anser,Greylag Goose,wetland,1.050000,-1756.500000,0.394683,0.009156,16
7,Grus grus,Common Crane,grassland_dryland,0.811765,-1274.985294,0.303176,0.027087,16
8,Grus virgo,Demoiselle Crane,grassland_dryland,1.294118,-2250.257353,0.546592,0.001065,16
9,Phoenicopterus roseus,Greater Flamingo,wetland,0.952941,-1560.808824,0.354481,0.014962,16


In [6]:
n_sig_arr = int(arrival_trend["p_value"].apply(stats_.is_significant).sum())
n_sig_dep = int(departure_trend["p_value"].apply(stats_.is_significant).sum())
print(f"Arrival trends significant at alpha={stats_.ALPHA}: {n_sig_arr}/{len(arrival_trend)}")
print(f"Departure trends significant at alpha={stats_.ALPHA}: {n_sig_dep}/{len(departure_trend)}")

Arrival trends significant at alpha=0.05: 12/12
Departure trends significant at alpha=0.05: 12/12


## Correlation analysis (H1): winter temperature vs arrival
Pearson r + p per species (Spearman shown as the secondary check).

In [7]:
corr = stats_.build_correlation_table(metrics, annual_env)
corr

,common_name,scientific_name,n_years,pearson_r,p_value,spearman_rho,interpretation
0,Northern Pintail,Anas acuta,16,0.335,0.2049,0.429,none
1,Northern Shoveler,Spatula clypeata,16,0.365,0.1646,0.414,none
2,Garganey,Spatula querquedula,16,0.461,0.0725,0.582,none
3,Eurasian Wigeon,Mareca penelope,16,0.580,0.0186,0.597,strong
4,Common Pochard,Aythya ferina,16,0.525,0.0369,0.550,strong
5,Bar-headed Goose,Anser indicus,16,0.572,0.0207,0.552,strong
6,Greylag Goose,Anser anser,16,0.500,0.0484,0.487,strong
7,Common Crane,Grus grus,16,0.331,0.2099,0.389,none
8,Demoiselle Crane,Grus virgo,16,0.502,0.0477,0.486,strong
9,Greater Flamingo,Phoenicopterus roseus,16,0.413,0.1120,0.346,none


## Habitat-category comparison (H3)
Category-level arrival trend: wetland-dependent vs grassland/dryland (comparative/descriptive, no formal group-difference test at this scope).

In [8]:
hab_year, hab_trends = stats_.habitat_category_trends(metrics)
for hab, tr in hab_trends.items():
    print(hab, "-> slope", None if tr["slope"] is None else round(tr["slope"],3),
          "days/yr, p =", None if tr["p_value"] is None else round(tr["p_value"],3))

grassland_dryland -> slope -0.975 days/yr, p = 0.0
wetland -> slope -0.698 days/yr, p = 0.0


## Hypothesis evaluation (H1-H3)
Decisions use alpha = 0.05 and are reported honestly per `04_STATISTICAL_ANALYSIS_PLAN.md`. H2 (timing) uses real eBird data; H1 (temperature) uses MOCK temperatures, so treat H1 as a placeholder.)

In [9]:
# Interpretation/assembly only -- all statistics come from src.
n_h1 = int((corr["interpretation"] != "none").sum())
h1 = "Supported" if n_h1 > len(corr)/2 else ("Inconclusive" if n_h1 else "Not Supported")
n_h2 = max(n_sig_arr, n_sig_dep)
h2 = "Supported" if n_h2 > len(arrival_trend)/2 else ("Inconclusive" if n_h2 else "Not Supported")
sw = hab_trends.get("wetland", {}).get("slope")
sd = hab_trends.get("grassland_dryland", {}).get("slope")
if sw is None or sd is None:
    h3 = "Inconclusive"
else:
    h3 = "Supported (categories differ)" if (np.sign(sw) != np.sign(sd) or abs(sw-sd) >= 1.0) else "Not Supported (similar)"
print(f"H1 (temperature -> earlier arrival): {h1}  ({n_h1}/{len(corr)} species significant)")
print(f"H2 (timing changed over period)   : {h2}  (max {n_h2}/{len(arrival_trend)} species significant)")
print(f"H3 (habitat categories differ)    : {h3}  (wetland {sw}, dryland {sd} days/yr)")

H1 (temperature -> earlier arrival): Inconclusive  (6/12 species significant)
H2 (timing changed over period)   : Supported  (max 12/12 species significant)
H3 (habitat categories differ)    : Not Supported (similar)  (wetland -0.6978758169934642, dryland -0.9745098039215686 days/yr)


_Conclusions above are computed live from the data. H2 uses real eBird observations; H1 uses MOCK temperatures (placeholder until GEE auth). A 'no meaningful association' outcome is a valid, reportable finding (`01_RESEARCH_METHODOLOGY.md`)._